# 🛡️ Phase 4 — Baseline Machine Learning Models

**F.R.A.U.D.S.H.I.E.L.D.** (*Financial Risk Assessment & Unauthorized Data Shield*)

---

## 🎯 Objective
In this phase, we establish empirical baseline performance by training and comparing **6 fundamental Machine Learning algorithms**:

1. **Logistic Regression** (Linear baseline)
2. **Decision Tree** (Single non-linear tree)
3. **Random Forest** (Ensemble of bagged decision trees)
4. **Gaussian Naive Bayes** (Probabilistic baseline)
5. **Support Vector Machine (SVM)** (Maximum margin separator with linear kernel)
6. **K-Nearest Neighbors (KNN)** (Instance-based non-parametric classifier)

### Hardware & Computational Trade-offs (16 GB RAM Rules):
- PaySim contains **6.36 million rows**. Training an RBF Support Vector Machine scales with $\mathcal{O}(N^2)$ to $\mathcal{O}(N^3)$ computational complexity, requiring days of compute and terabytes of intermediate kernel memory.
- Similarly, KNN requires storing all training points and computing pairwise distances at test time ($\mathcal{O}(N \times D)$).
- To maintain fast, responsive execution without out-of-memory (OOM) crashes, we train on a representative, **stratified subsample** (e.g. 50,000–100,000 records) preserving the rare ~0.13% fraud proportion.

## 1. Setup & Environment Imports

In [ ]:
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project path configuration
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, MODELS_DIR
from src.data.loader import find_paysim_file, load_dataset
from src.data.preprocessor import FraudDataPreprocessor, split_data
from src.models.trainer import BaselineModelTrainer, get_baseline_models

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Environment initialized successfully.")

## 2. Ingest Dataset & Stratified Splitting
We load a representative sample, split into Train (80%) and Test (20%) using **stratification** (so both splits contain proportional fraud), and apply our Phase 2 `FraudDataPreprocessor`.

In [ ]:
dataset_file = find_paysim_file()
if dataset_file and dataset_file.exists():
    print(f"Loading 100,000 stratified records from {dataset_file}...")
    df = load_dataset(dataset_file, nrows=100000, use_optimized_dtypes=True)
else:
    print("Using representative PaySim schema sample (50,000 rows)...")
    np.random.seed(42)
    n = 50000
    types = np.random.choice(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'], size=n, p=[0.34, 0.08, 0.35, 0.01, 0.22])
    amounts = np.random.exponential(scale=150000, size=n).astype(np.float32)
    old_bal_org = np.random.exponential(scale=800000, size=n).astype(np.float32)
    new_bal_org = np.maximum(0, old_bal_org - amounts)
    old_bal_dest = np.random.exponential(scale=1000000, size=n).astype(np.float32)
    new_bal_dest = old_bal_dest + amounts
    
    fraud_mask = (types == 'TRANSFER') & (amounts > 200000) & (np.random.rand(n) < 0.05)
    fraud_mask |= (types == 'CASH_OUT') & (amounts > 150000) & (np.random.rand(n) < 0.03)
    is_fraud = fraud_mask.astype(np.uint8)
    
    df = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 745, size=n, dtype=np.uint16)),
        'type': pd.Categorical(types),
        'amount': amounts,
        'nameOrig': [f'C{i:09d}' for i in range(n)],
        'oldbalanceOrg': old_bal_org,
        'newbalanceOrig': new_bal_org,
        'nameDest': [f'M{i:09d}' if t == 'PAYMENT' else f'C{i+50000:09d}' for i, t in enumerate(types)],
        'oldbalanceDest': old_bal_dest,
        'newbalanceDest': new_bal_dest,
        'isFraud': is_fraud,
        'isFlaggedFraud': np.zeros(n, dtype=np.uint8)
    })

# 1. Split strictly before preprocessing
X_train_raw, X_test_raw, y_train, y_test = split_data(
    df, target_col="isFraud", test_size=0.20, split_strategy="stratified", random_state=42
)

# 2. Fit preprocessor on X_train only
preprocessor = FraudDataPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"X_train shape: {X_train.shape} | y_train fraud cases: {y_train.sum():,}")
print(f"X_test shape:  {X_test.shape}  | y_test fraud cases:  {y_test.sum():,}")

## 3. Algorithmic Overview: The 6 Baseline Classifiers

| Algorithm | Type | Computational Complexity | Fraud Detection Strengths & Weaknesses |
| :--- | :--- | :--- | :--- |
| **Logistic Regression** | Linear Probabilistic | $\mathcal{O}(N \times D)$ | Very fast, highly interpretable. Weak on complex non-linear balance interactions. |
| **Decision Tree** | Non-Linear Partitioning | $\mathcal{O}(D \times N \log N)$ | Excellent at threshold logic (e.g. `amount > X` AND `errorBalance == Y`). Prone to overfitting without depth limits. |
| **Random Forest** | Ensemble Bagging | $\mathcal{O}(M \times D \times N \log N)$ | Industry gold-standard for tabular data. Averages out individual tree variance. Higher training compute. |
| **Naive Bayes** | Generative Probabilistic | $\mathcal{O}(N \times D)$ | Extremely fast. Assumes feature independence, which is violated by correlated balances. |
| **Support Vector Machine (Linear)** | Maximum Margin Separator | $\mathcal{O}(N \times D)$ | Effective in high-dimensional feature spaces. Calibrated via Platt scaling to output probabilities. |
| **K-Nearest Neighbors** | Instance-Based (Lazy) | $\mathcal{O}(N \times D)$ per test sample | Intuitive similarity metric. Computationally slow at test time because every prediction queries the dataset. |

## 4. Train All 6 Baseline Models

In [ ]:
trainer = BaselineModelTrainer(random_state=42)
print("Training all 6 baseline models on X_train...\n")

for model_name in get_baseline_models().keys():
    print(f"--> Training {model_name}...")
    trainer.train_model(model_name, X_train, y_train)
    duration = trainer.training_times[model_name]
    print(f"    Completed in {duration:.3f} seconds.")

print("\nAll 6 baseline models trained successfully!")

## 5. Evaluate Inference Latency & Prediction Outputs
In production financial systems, latency is critical: transactions must be authorized within 50–100 milliseconds.
Let's measure inference speed on the test set.

In [ ]:
inference_results = []

for model_name in trainer.trained_models.keys():
    y_pred, y_prob, pred_time = trainer.predict(model_name, X_test)
    
    # Measure latency per 1,000 transactions (in ms)
    latency_per_1k = (pred_time / len(X_test)) * 1000 * 1000
    
    inference_results.append({
        "Model": model_name,
        "Train Time (s)": trainer.training_times[model_name],
        "Inference Time (s)": pred_time,
        "Latency per 1K Tx (ms)": latency_per_1k,
        "Fraud Predictions Count": int(y_pred.sum()),
        "Mean Fraud Probability": float(y_prob.mean())
    })

inference_df = pd.DataFrame(inference_results)
display(inference_df)

## 6. Visualizing Training and Inference Latency Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.barplot(data=inference_df, x="Train Time (s)", y="Model", ax=axes[0], palette="Blues_r", hue="Model", legend=False)
axes[0].set_title("Training Duration (Seconds)", fontweight="bold")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

sns.barplot(data=inference_df, x="Latency per 1K Tx (ms)", y="Model", ax=axes[1], palette="Purples_r", hue="Model", legend=False)
axes[1].set_title("Inference Latency per 1,000 Transactions (ms)", fontweight="bold")
axes[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Model Persistence: Save Baseline Artifacts to `models/`

In [ ]:
print("Saving trained baseline models to disk:")
for model_name in trainer.trained_models.keys():
    saved_path = trainer.save_model(model_name)
    file_size_kb = saved_path.stat().st_size / 1024
    print(f"  [Saved] {model_name:25s} -> {saved_path.name} ({file_size_kb:,.1f} KB)")

## 8. Phase 4 Summary & Next Steps

| Algorithm | Implementation Detail | Role in Fraud Platform |
| :--- | :--- | :--- |
| **Logistic Regression** | `solver='lbfgs'`, L2 regularization | Simple linear benchmark for latency and explainability. |
| **Decision Tree** | `max_depth=12`, `min_samples_split=20` | Fast non-linear tree rules; base learner for tree ensembles. |
| **Random Forest** | `n_estimators=100`, parallel multi-core | Premier non-linear ensemble candidate for accuracy and stability. |
| **Naive Bayes** | `GaussianNB` | Generative baseline; fastest training speed. |
| **Support Vector Machine** | `LinearSVC` with Platt scaling calibration | Maximum-margin linear classifier with calibrated probabilities. |
| **K-Nearest Neighbors** | `n_neighbors=5`, Euclidean distance | Local geometric similarity classifier; higher inference latency. |

---
**Phase 4 Complete:** All 6 baseline models are trained, evaluated for latency, and persisted to disk. In Phase 5, we will compute comprehensive evaluation metrics (Precision, Recall, F1, ROC-AUC, PR-AUC, Confusion Matrices).